# House Pricing Analysis with MLP Regression

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Load Datasets

In [ ]:
with open("../datasets/house_prices.pkl", mode="rb") as f:
    X_house_prices_train, X_house_prices_test, y_house_prices_train, y_house_prices_test = pickle.load(f)

print(X_house_prices_train.shape)
print(X_house_prices_test.shape)
print(y_house_prices_train.shape)
print(y_house_prices_test.shape)

## Tuning

### Grid Search

In [ ]:
pipeline_base = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor())
])

wrapper_model = TransformedTargetRegressor(
    regressor=pipeline_base,
    func=np.log1p,
    inverse_func=np.expm1
)

params = {
    'regressor__mlp__max_iter': [500],
    'regressor__mlp__early_stopping': [True],
    'regressor__mlp__hidden_layer_sizes': [(32, 16), (64, 32),(64, 32, 16)],
    'regressor__mlp__activation': ['relu'],
    'regressor__mlp__solver': ['adam'],
    'regressor__mlp__alpha': [0.0001, 0.001, 0.01],
    'regressor__mlp__learning_rate_init': [0.001, 0.01]
}

grid_search = GridSearchCV(
    estimator=wrapper_model,
    param_grid=params,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
).fit(
    X=X_house_prices_train,
    y=y_house_prices_train
)

print(f"Best params: {grid_search.best_params_}")
# {'regressor__mlp__activation': 'relu', 'regressor__mlp__alpha': 0.01, 'regressor__mlp__early_stopping': True, 'regressor__mlp__hidden_layer_sizes': (64, 32), 'regressor__mlp__learning_rate_init': 0.001, 'regressor__mlp__max_iter': 500, 'regressor__mlp__solver': 'adam'}

## Train/Test

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPRegressor(
        activation='relu',
        early_stopping=True,
        max_iter=500,
        alpha=0.01,
        hidden_layer_sizes=(64, 32),
        learning_rate_init=0.001,
        solver='adam'
    ))
])

model = TransformedTargetRegressor(
    regressor=pipeline,
    func=np.log1p,
    inverse_func=np.expm1
).fit(
    X=X_house_prices_train,
    y=y_house_prices_train
)

## Evaluation

### Metrics

In [ ]:
print(f"R²: {model.score(X=X_house_prices_train, y=y_house_prices_train)} (train set)")
print(f"R²: {model.score(X=X_house_prices_test, y=y_house_prices_test)} (test set)")

In [ ]:
y_house_prices_pred = model.predict(X_house_prices_test)

print(f"MAE: {mean_absolute_error(y_pred=y_house_prices_pred, y_true=y_house_prices_test)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_pred=y_house_prices_pred, y_true=y_house_prices_test))}")

### Visualization

In [ ]:
plt.figure(figsize=(10, 10))
plt.title(f"Estimator's Overlay of True and Predict Values for MLP Regressor")
plt.plot(y_house_prices_test[::50], label="Y Test Set")
plt.plot(y_house_prices_pred[::50], label="Y Pred Set")
plt.legend();